In [1]:
import torch
import copy
from accelerate import Accelerator
from torch.utils.data import Dataset, DataLoader



class MyDataset(Dataset):
    def __init__(self, num_samples):
        super().__init__()
        self.len = num_samples

    def __getitem__(self, index):
	# input_ids – [1, 2, …, index+1], labels = [1, 0, 1, 0, …, (index+1) % 2]
        input_ids = torch.arange(1, index+2, dtype=torch.float32)
        labels = torch.remainder(input_ids, 2)  # остаток от деления на 2
        return {"input_ids": input_ids, "labels": labels}

    def __len__(self):
        return self.len
    
def collate_fn(features):
    input_ids = torch.nn.utils.rnn.pad_sequence([f["input_ids"] for f in features], 		batch_first=True, padding_value=-100)
    labels = torch.nn.utils.rnn.pad_sequence([f["labels"] for f in features], 
	batch_first=True, padding_value=-100)
    return {"input_ids": input_ids[..., None], "labels": labels[..., None]}


# define toy inputs and labels
gradient_accumulation_steps = 2
per_device_batch_size = 4

# define accelerator
accelerator = Accelerator(gradient_accumulation_steps=gradient_accumulation_steps)

# define dataset and dataloader
# for this toy example, we'll compute gradient descent over one single global batch
dataset = MyDataset(per_device_batch_size*gradient_accumulation_steps*accelerator.num_processes)
dataloader = DataLoader(dataset, batch_size=per_device_batch_size, collate_fn=collate_fn)


In [2]:
dataloader = accelerator.prepare_data_loader(dataloader, device_placement=True)

In [3]:
training_iterator = iter(dataloader)

In [5]:
next(training_iterator)

{'input_ids': tensor([[[   1.],
          [-100.],
          [-100.],
          [-100.]],
 
         [[   1.],
          [   2.],
          [-100.],
          [-100.]],
 
         [[   1.],
          [   2.],
          [   3.],
          [-100.]],
 
         [[   1.],
          [   2.],
          [   3.],
          [   4.]]], device='cuda:0'),
 'labels': tensor([[[   1.],
          [-100.],
          [-100.],
          [-100.]],
 
         [[   1.],
          [   0.],
          [-100.],
          [-100.]],
 
         [[   1.],
          [   0.],
          [   1.],
          [-100.]],
 
         [[   1.],
          [   0.],
          [   1.],
          [   0.]]], device='cuda:0')}

In [6]:
next(training_iterator)

{'input_ids': tensor([[[   1.],
          [   2.],
          [   3.],
          [   4.],
          [   5.],
          [-100.],
          [-100.],
          [-100.]],
 
         [[   1.],
          [   2.],
          [   3.],
          [   4.],
          [   5.],
          [   6.],
          [-100.],
          [-100.]],
 
         [[   1.],
          [   2.],
          [   3.],
          [   4.],
          [   5.],
          [   6.],
          [   7.],
          [-100.]],
 
         [[   1.],
          [   2.],
          [   3.],
          [   4.],
          [   5.],
          [   6.],
          [   7.],
          [   8.]]], device='cuda:0'),
 'labels': tensor([[[   1.],
          [   0.],
          [   1.],
          [   0.],
          [   1.],
          [-100.],
          [-100.],
          [-100.]],
 
         [[   1.],
          [   0.],
          [   1.],
          [   0.],
          [   1.],
          [   0.],
          [-100.],
          [-100.]],
 
         [[   1.],
          [